In [ ]:
!pip install torch-geometric


In [ ]:
"""
FRAANet: Feature-Refined Attention & Aggregation Network
For Anti-Money Laundering Detection in Cryptocurrency Networks.

Target Journal: Q1 (e.g., IEEE TIFS, Expert Systems with Applications)

"""

import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, ReLU, Sequential, Dropout, LayerNorm
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, SAGEConv
from torch_geometric.utils import k_hop_subgraph, dropout_edge
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, f1_score)
from sklearn.manifold import TSNE
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ============================================================
CONFIG = {
    # File Paths
    'classes_path':  '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_classes.csv',
    'edges_path':    '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv',
    'features_path': '/kaggle/input/datasets/organizations/ellipticco/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_features.csv',

    # Save path
    'save_dir': '/kaggle/working/',

    # Model Hyperparameters
    'in_channels':      165,
    'hidden_channels':  48,
    'out_channels':     1,
    'attention_heads':  4,
    'dropout_rate':     0.5,

    # Training
    'epochs':           200,
    'learning_rate':    0.001,
    'weight_decay':     5e-4,
    'early_stop_patience': 30,

    # Focal Loss
    'focal_alpha': 0.75,
    'focal_gamma': 2.0,
    'label_smoothing': 0.05,

    # DropEdge — randomly drops edges during training (graph-level dropout)
    'drop_edge_p': 0.15,

    # Mini-batch settings (no pyg-lib needed)
    'batch_size':    256,
    'num_hops':      2,

    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)


# ============================================================
# 2. DATA PREPROCESSING
# ============================================================
def load_elliptic_data(config):
    print("Loading datasets...")
    df_classes  = pd.read_csv(config['classes_path'])
    df_edges    = pd.read_csv(config['edges_path'])
    df_features = pd.read_csv(config['features_path'], header=None)

    col_names = ['txId', 'time_step'] + [f'feat_{i}' for i in range(config['in_channels'])]
    df_features.columns = col_names

    df = df_features.merge(df_classes, on='txId', how='left')
    df['class'] = df['class'].map({'1': 1, '2': 0, 'unknown': -1})
    df = df.sort_values('time_step').reset_index(drop=True)

    mapping = {txid: i for i, txid in enumerate(df['txId'].values)}
    df_edges['txId1'] = df_edges['txId1'].map(mapping)
    df_edges['txId2'] = df_edges['txId2'].map(mapping)
    df_edges = df_edges.dropna()

    edge_index = torch.tensor(df_edges[['txId1', 'txId2']].values.T, dtype=torch.long)
    x          = torch.tensor(df.drop(columns=['txId', 'time_step', 'class']).values, dtype=torch.float)
    y          = torch.tensor(df['class'].values, dtype=torch.long)
    time_steps = torch.tensor(df['time_step'].values, dtype=torch.long)

    train_mask = (time_steps <= 34) & (y != -1)
    val_mask   = ((time_steps > 34) & (time_steps <= 39)) & (y != -1)
    test_mask  = (time_steps > 39)  & (y != -1)

    data = Data(x=x, edge_index=edge_index, y=y)
    data.train_mask = train_mask
    data.val_mask   = val_mask
    data.test_mask  = test_mask

    feature_names = df.drop(columns=['txId', 'time_step', 'class']).columns.tolist()

    print(f"Data Loaded: {data.num_nodes} nodes, {data.num_edges} edges.")
    n_ill = (y[train_mask] == 1).sum().item()
    n_lic = (y[train_mask] == 0).sum().item()
    print(f"Train set  → Licit: {n_lic}  |  Illicit: {n_ill}  |  Ratio: {n_lic/n_ill:.1f}:1")

    return data, feature_names


# ============================================================
# 3. CUSTOM MINI-BATCH SAMPLER  (no pyg-lib / torch-sparse)
# ============================================================
class SubgraphBatchSampler:
    """
    Splits seed node indices into mini-batches.
    Extracts k-hop subgraph using torch_geometric.utils.k_hop_subgraph
    (pure Python/PyTorch — no C++ extension required).
    """
    def __init__(self, node_indices, batch_size, num_hops, edge_index,
                 num_nodes, shuffle=True):
        self.node_indices = node_indices
        self.batch_size   = batch_size
        self.num_hops     = num_hops
        self.edge_index   = edge_index   # CPU
        self.num_nodes    = num_nodes
        self.shuffle      = shuffle

    def __iter__(self):
        idx = self.node_indices.clone()
        if self.shuffle:
            perm = torch.randperm(len(idx))
            idx  = idx[perm]

        for start in range(0, len(idx), self.batch_size):
            seeds = idx[start: start + self.batch_size]
            # k_hop_subgraph returns 4 values: subset, sub_edge_index, mapping, edge_mask
            subset, sub_edge_index, mapping, _ = k_hop_subgraph(
                node_idx=seeds,
                num_hops=self.num_hops,
                edge_index=self.edge_index,
                relabel_nodes=True,
                num_nodes=self.num_nodes,
                flow='source_to_target'
            )
            yield subset, sub_edge_index, mapping

    def __len__(self):
        return (len(self.node_indices) + self.batch_size - 1) // self.batch_size


# ============================================================
# 4. FOCAL LOSS WITH LABEL SMOOTHING
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Apply label smoothing to soft targets
        smooth_targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce   = F.binary_cross_entropy_with_logits(logits, smooth_targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets >= 0.5, probs, 1.0 - probs)
        alpha = torch.where(targets >= 0.5,
                            torch.full_like(targets, self.alpha),
                            torch.full_like(targets, 1.0 - self.alpha))
        return (alpha * ((1.0 - pt) ** self.gamma) * bce).mean()


# ============================================================
# 5. ARCHITECTURE COMPONENTS
# ============================================================
class ResidualMLP(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.net  = Sequential(
            Linear(in_dim, out_dim), LayerNorm(out_dim), ReLU(), Dropout(dropout),
            Linear(out_dim, out_dim), LayerNorm(out_dim),
        )
        self.proj = Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.act  = ReLU()

    def forward(self, x):
        return self.act(self.net(x) + self.proj(x))


class GraphAttentionAggregator(nn.Module):
    """
    Stacked GATv2 → GraphSAGE with skip connections.
    Captures multi-hop relational context.
    """
    def __init__(self, hidden, heads, dropout):
        super().__init__()
        self.gat1  = GATv2Conv(hidden, hidden, heads=heads, concat=False, dropout=dropout)
        self.gat2  = GATv2Conv(hidden, hidden, heads=heads, concat=False, dropout=dropout)
        self.sage  = SAGEConv(hidden, hidden)
        self.norm1 = LayerNorm(hidden)
        self.norm2 = LayerNorm(hidden)
        self.norm3 = LayerNorm(hidden)
        self.drop  = Dropout(dropout)

    def forward(self, x, edge_index):
        h1 = self.norm1(F.elu(self.gat1(x,  edge_index)) + x)
        h1 = self.drop(h1)
        h2 = self.norm2(F.elu(self.gat2(h1, edge_index)) + h1)
        h2 = self.drop(h2)
        h3 = self.norm3(F.elu(self.sage(h2, edge_index)) + h2)
        return h1, h2, h3


# ============================================================
# 6. FRAANET
# ============================================================
class FRAANet(torch.nn.Module):
    """
    Feature-Refined Attention & Aggregation Network (FRAANet).

    Novelty:
    1. Feature Refinement Bottleneck with residual MLP — denoises anonymized features.
    2. GraphAttentionAggregator — stacked GATv2 + SAGE with skip connections.
    3. Jumping Knowledge (JK) Fusion — concatenates ALL latent representations.
    4. Focal Loss with label smoothing — handles severe 2% illicit imbalance.
    5. DropEdge during training — acts as graph-level stochastic regularization.
    """
    def __init__(self, in_channels, hidden_channels, out_channels, heads, dropout_rate):
        super().__init__()
        h = hidden_channels

        self.frm_in  = ResidualMLP(in_channels, h * 2, dropout_rate)
        self.frm_out = ResidualMLP(h * 2,       h,     dropout_rate)
        self.gaa     = GraphAttentionAggregator(h, heads, dropout_rate)

        # JK concat: x_ref + h1 + h2 + h3 = 4 × h
        self.classifier = Sequential(
            Linear(h * 4, h * 2), LayerNorm(h * 2), ReLU(), Dropout(dropout_rate),
            Linear(h * 2, h),     LayerNorm(h),     ReLU(), Dropout(dropout_rate),
            Linear(h, out_channels)
        )

    def forward(self, x, edge_index, drop_edge_p=0.0):
        # DropEdge: randomly remove edges during training for regularization
        if drop_edge_p > 0.0 and self.training:
            edge_index, _ = dropout_edge(edge_index, p=drop_edge_p, training=True)
        x_ref      = self.frm_out(self.frm_in(x))
        h1, h2, h3 = self.gaa(x_ref, edge_index)
        return self.classifier(torch.cat([x_ref, h1, h2, h3], dim=1))

    def get_embeddings(self, x, edge_index):
        with torch.no_grad():
            x_ref      = self.frm_out(self.frm_in(x))
            h1, h2, h3 = self.gaa(x_ref, edge_index)
            return torch.cat([x_ref, h1, h2, h3], dim=1)


# ============================================================
# 7. TRAINING & EVALUATION
# ============================================================
def run_epoch_train(model, sampler, data, criterion, optimizer, device, drop_edge_p):
    model.train()
    total_loss, total_n = 0.0, 0
    all_logits, all_labels = [], []

    for subset, sub_ei, mapping in sampler:
        sub_x  = data.x[subset].to(device)
        sub_y  = data.y[subset].to(device)
        sub_ei = sub_ei.to(device)

        optimizer.zero_grad()
        out      = model(sub_x, sub_ei, drop_edge_p=drop_edge_p).squeeze()
        out_seed = out[mapping]
        y_seed   = sub_y[mapping].float()

        valid = (y_seed != -1)
        if valid.sum() == 0:
            continue

        loss = criterion(out_seed[valid], y_seed[valid])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        n = valid.sum().item()
        total_loss += loss.item() * n
        total_n    += n
        all_logits.append(out_seed[valid].detach().cpu())
        all_labels.append(y_seed[valid].long().cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    preds  = (torch.sigmoid(logits) > 0.5).long().numpy()
    f1     = f1_score(labels.numpy(), preds, average='macro', zero_division=0)
    return total_loss / max(total_n, 1), f1


@torch.no_grad()
def run_epoch_eval(model, sampler, data, criterion, device):
    model.eval()
    total_loss, total_n = 0.0, 0
    all_logits, all_labels = [], []

    for subset, sub_ei, mapping in sampler:
        sub_x  = data.x[subset].to(device)
        sub_y  = data.y[subset].to(device)
        sub_ei = sub_ei.to(device)

        out      = model(sub_x, sub_ei, drop_edge_p=0.0).squeeze()
        out_seed = out[mapping]
        y_seed   = sub_y[mapping].float()

        valid = (y_seed != -1)
        if valid.sum() == 0:
            continue

        loss = criterion(out_seed[valid], y_seed[valid])
        n    = valid.sum().item()
        total_loss += loss.item() * n
        total_n    += n
        all_logits.append(out_seed[valid].cpu())
        all_labels.append(y_seed[valid].long().cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs  = torch.sigmoid(logits).numpy()
    preds  = (probs > 0.5).astype(int)
    f1     = f1_score(labels.numpy(), preds, average='macro', zero_division=0)
    return total_loss / max(total_n, 1), f1, labels.numpy(), preds, probs


def train_model(model, data, config):
    print("\nBuilding mini-batch samplers (no pyg-lib required)...")
    train_idx = torch.where(data.train_mask)[0]
    val_idx   = torch.where(data.val_mask)[0]

    train_sampler = SubgraphBatchSampler(
        train_idx, config['batch_size'], config['num_hops'],
        data.edge_index, data.num_nodes, shuffle=True)
    val_sampler   = SubgraphBatchSampler(
        val_idx, config['batch_size'] * 2, config['num_hops'],
        data.edge_index, data.num_nodes, shuffle=False)

    criterion = FocalLoss(config['focal_alpha'], config['focal_gamma'],
                          config['label_smoothing'])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config['learning_rate'],
                                  weight_decay=config['weight_decay'])

    # Warmup for 10 epochs then cosine decay — prevents early aggressive overfitting
    def lr_lambda(epoch):
        warmup = 10
        if epoch < warmup:
            return float(epoch + 1) / warmup
        progress = (epoch - warmup) / max(1, config['epochs'] - warmup)
        return max(0.05, 0.5 * (1.0 + np.cos(np.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {'train_loss': [], 'val_loss': [],
               'train_f1_macro': [], 'val_f1_macro': []}

    best_val_f1    = 0.0
    best_weights   = None
    patience_count = 0

    print("\nInitializing Training...\n")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'Train F1':>9} | {'Val F1':>8}")
    print("-" * 58)

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_f1 = run_epoch_train(
            model, train_sampler, data, criterion, optimizer,
            config['device'], config['drop_edge_p'])
        val_loss, val_f1, _, _, _ = run_epoch_eval(
            model, val_sampler, data, criterion, config['device'])
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1_macro'].append(train_f1)
        history['val_f1_macro'].append(val_f1)

        # Print every epoch
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>8.4f} | "
              f"{train_f1:>9.4f} | {val_f1:>8.4f}")

        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        # Early stopping
        if patience_count >= config['early_stop_patience']:
            print(f"\nEarly stopping at epoch {epoch} — "
                  f"no improvement for {config['early_stop_patience']} epochs.")
            break

        if epoch % 10 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    if best_weights:
        model.load_state_dict(best_weights)
        print(f"\nBest model restored — Val F1 (macro): {best_val_f1:.4f}")

    return model, history


def evaluate_model(model, data, config):
    print("\nEvaluating on Test Set...")
    test_idx     = torch.where(data.test_mask)[0]
    test_sampler = SubgraphBatchSampler(
        test_idx, config['batch_size'] * 2, config['num_hops'],
        data.edge_index, data.num_nodes, shuffle=False)
    criterion = FocalLoss(config['focal_alpha'], config['focal_gamma'],
                          config['label_smoothing'])

    _, _, y_true, preds, probs = run_epoch_eval(
        model, test_sampler, data, criterion, config['device'])

    print("\n" + "=" * 50)
    print("FINAL TEST SET EVALUATION")
    print("=" * 50)
    print(classification_report(y_true, preds, digits=4,
                                 target_names=['Licit (0)', 'Illicit (1)']))
    return y_true, preds, probs


# ============================================================
# 8. INDIVIDUAL METRIC PLOTS
# ============================================================
def save_fig(fname, config):
    path = os.path.join(config['save_dir'], fname)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Saved: {path}")


def plot_train_val_loss(history, config):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history['train_loss'], label='Train Loss',  color='steelblue',  linewidth=2)
    ax.plot(history['val_loss'],   label='Val Loss',    color='darkorange', linewidth=2)
    ax.set_title('Training & Validation Loss', fontsize=15, weight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Focal Loss')
    ax.legend(fontsize=12)
    plt.tight_layout()
    save_fig('plot1_loss_curve.png', config)


def plot_train_val_f1(history, config):
    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history['train_f1_macro'], label='Training F1 (macro)',   color='mediumseagreen', linewidth=2)
    ax.plot(history['val_f1_macro'],   label='Validation F1 (macro)', color='tomato',         linewidth=2)
    ax.set_title('Training & Validation F1 Score (Macro)', fontsize=15, weight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score (macro)')
    ax.legend(fontsize=12)
    plt.tight_layout()
    save_fig('plot2_f1_curve.png', config)


def plot_confusion_matrix(y_true, preds, config):
    sns.set_theme(style="white")
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(y_true, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                annot_kws={"size": 16}, ax=ax)
    ax.set_title('Confusion Matrix', fontsize=15, weight='bold')
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
    ax.set_xticklabels(['Licit', 'Illicit'])
    ax.set_yticklabels(['Licit', 'Illicit'])
    plt.tight_layout()
    save_fig('plot3_confusion_matrix.png', config)


def plot_roc_curve(y_true, probs, config):
    fig, ax = plt.subplots(figsize=(6, 5))
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
    ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax.set_title('ROC Curve', fontsize=15, weight='bold')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.legend(loc="lower right", fontsize=12)
    plt.tight_layout()
    save_fig('plot4_roc_curve.png', config)


def plot_pr_curve(y_true, probs, config):
    fig, ax = plt.subplots(figsize=(6, 5))
    precision, recall, _ = precision_recall_curve(y_true, probs)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, color='purple', lw=2, label=f'PR AUC = {pr_auc:.4f}')
    ax.set_title('Precision-Recall Curve', fontsize=15, weight='bold')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(loc="lower left", fontsize=12)
    plt.tight_layout()
    save_fig('plot5_pr_curve.png', config)


# ============================================================
# 9. t-SNE CLASSIFICATION VISUALIZATION  (fixed)
# ============================================================
def plot_tsne_classification(model, data, config, n_samples=3000):
    """
    t-SNE of learned embeddings.
    Purple = Licit (0), Green = Illicit (1).

    FIX: k_hop_subgraph returns 4 values — correctly unpack all 4.
    """
    print("\nGenerating t-SNE classification visualization...")
    model.eval()
    device = config['device']

    test_idx  = torch.where(data.test_mask)[0]
    n_samples = min(n_samples, len(test_idx))
    rng       = np.random.default_rng(42)
    chosen    = rng.choice(len(test_idx), size=n_samples, replace=False)
    chosen_idx = test_idx[chosen]

    all_embs, all_labels = [], []
    batch_size = 128
    for start in range(0, len(chosen_idx), batch_size):
        seeds = chosen_idx[start: start + batch_size]

        # FIXED: unpack all 4 return values from k_hop_subgraph
        subset, sub_ei, mapping, _ = k_hop_subgraph(
            node_idx=seeds,
            num_hops=config['num_hops'],
            edge_index=data.edge_index,
            relabel_nodes=True,
            num_nodes=data.num_nodes,
            flow='source_to_target'
        )
        sub_x  = data.x[subset].to(device)
        sub_ei = sub_ei.to(device)
        emb    = model.get_embeddings(sub_x, sub_ei)
        all_embs.append(emb[mapping].cpu())
        all_labels.append(data.y[seeds])
        torch.cuda.empty_cache()

    emb_np = torch.cat(all_embs).numpy()
    lbl_np = torch.cat(all_labels).numpy()

    tsne   = TSNE(n_components=2, perplexity=40, n_iter=1000,
                  random_state=42, init='pca', learning_rate='auto')
    coords = tsne.fit_transform(emb_np)

    color_map = {0: '#7B2D8B', 1: '#3CB371'}
    colors    = [color_map.get(int(l), '#AAAAAA') for l in lbl_np]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.75, linewidths=0)
    legend_handles = [
        mpatches.Patch(color='#7B2D8B', label='Licit (Class 0)'),
        mpatches.Patch(color='#3CB371', label='Illicit (Class 1)'),
    ]
    ax.legend(handles=legend_handles, fontsize=10, loc='upper right')
    ax.set_title('Classification visualization\n(FRAANet)', fontsize=13, weight='bold')
    ax.tick_params(labelsize=9)
    plt.tight_layout()
    save_fig('plot6_tsne_visualization.png', config)


# ============================================================
# 11. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":

    # Load Data (stays on CPU; only mini-batches go to GPU)
    data, feature_names = load_elliptic_data(CONFIG)

    # Initialize Model
    model = FRAANet(
        in_channels=CONFIG['in_channels'],
        hidden_channels=CONFIG['hidden_channels'],
        out_channels=CONFIG['out_channels'],
        heads=CONFIG['attention_heads'],
        dropout_rate=CONFIG['dropout_rate']
    ).to(CONFIG['device'])

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"FRAANet parameters: {total_params:,}")

    # Train
    model, history = train_model(model, data, CONFIG)

    # Evaluate
    y_true, preds, probs = evaluate_model(model, data, CONFIG)

    # Individual Plots
    plot_train_val_loss(history, CONFIG)
    plot_train_val_f1(history, CONFIG)
    plot_confusion_matrix(y_true, preds, CONFIG)
    plot_roc_curve(y_true, probs, CONFIG)
    plot_pr_curve(y_true, probs, CONFIG)

    # t-SNE Classification Visualization
    plot_tsne_classification(model, data, CONFIG)

    print(f"\nAll outputs saved to: {CONFIG['save_dir']}")

Loading datasets...
Data Loaded: 203769 nodes, 234355 edges.
Train set  → Licit: 26432  |  Illicit: 3462  |  Ratio: 7.6:1
FRAANet parameters: 120,001

Building mini-batch samplers (no pyg-lib required)...

Initializing Training...

 Epoch | Train Loss | Val Loss |  Train F1 |   Val F1
----------------------------------------------------------
     1 |     0.0525 |   0.0268 |    0.5174 |   0.7953
     2 |     0.0380 |   0.0287 |    0.6646 |   0.6563
     3 |     0.0247 |   0.0283 |    0.8179 |   0.6977
     4 |     0.0210 |   0.0285 |    0.8655 |   0.7299
     5 |     0.0183 |   0.0249 |    0.8861 |   0.7842
     6 |     0.0158 |   0.0279 |    0.8994 |   0.7925
     7 |     0.0143 |   0.0283 |    0.9112 |   0.7974
     8 |     0.0134 |   0.0314 |    0.9109 |   0.7961
     9 |     0.0126 |   0.0271 |    0.9179 |   0.7963
    10 |     0.0119 |   0.0148 |    0.9203 |   0.8350
    11 |     0.0111 |   0.0135 |    0.9283 |   0.8513
    12 |     0.0106 |   0.0155 |    0.9335 |   0.8260
    13 

In [ ]:
# ============================================================
# 10. XAI MODULE — LIME & SHAP
# ============================================================
def _build_xai_wrapper(model, data, target_node_idx, device, num_hops):
    """
    Perturbation wrapper for LIME/SHAP.
    Extracts k-hop subgraph around target node (pure PyG, no pyg-lib).
    """
    # Pre-compute the fixed subgraph topology once
    subset, sub_ei, mapping_tensor, _ = k_hop_subgraph(
        node_idx=torch.tensor([target_node_idx]),
        num_hops=num_hops,
        edge_index=data.edge_index,
        relabel_nodes=True,
        num_nodes=data.num_nodes,
        flow='source_to_target'
    )
    # Position of target node inside the subgraph
    target_pos = mapping_tensor[0].item()
    sub_ei_dev = sub_ei.to(device)

    def gnn_predict_wrapper(node_features_array):
        probs_list =[]
        # batch all perturbations together for speed
        base_x = data.x[subset].clone()   # (sub_size, 165)  on CPU

        for perturbed_feat in node_features_array:
            temp_x               = base_x.clone()
            temp_x[target_pos]   = torch.tensor(perturbed_feat, dtype=torch.float)
            temp_x_dev           = temp_x.to(device)

            with torch.no_grad():
                out  = model(temp_x_dev, sub_ei_dev).squeeze()
                logit = out[target_pos] if out.dim() > 0 else out
                prob  = torch.sigmoid(logit).item()
            probs_list.append([1 - prob, prob])

        return np.array(probs_list)

    return gnn_predict_wrapper


def explain_with_lime(wrapper, bg_data, target_feat, feature_names,
                      target_node_idx, example_id, config):
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=bg_data,
        feature_names=feature_names,
        class_names=['Licit', 'Illicit'],
        mode='classification'
    )
    exp = explainer.explain_instance(
        data_row=target_feat[0], predict_fn=wrapper, num_features=10)

    top_features = exp.as_list(label=1)
    names   = [f[0] for f in top_features]
    weights = [f[1] for f in top_features]
    colors_bar =['#E74C3C' if w > 0 else '#2ECC71' for w in weights]

    # Style A: Full bar
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(names, weights, color=colors_bar)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'LIME — Style A: Full Feature Weights\n(Node {target_node_idx}, Example {example_id})',
                 fontsize=13, weight='bold')
    ax.set_xlabel('Feature Weight')
    legend =[mpatches.Patch(color='#E74C3C', label='→ Illicit'),
              mpatches.Patch(color='#2ECC71', label='→ Licit')]
    ax.legend(handles=legend)
    plt.tight_layout()
    save_fig(f'lime_styleA_ex{example_id}.png', config)

    # Style B: Illicit-pushing features
    pos_names   =[n for n, w in zip(names, weights) if w > 0]
    pos_weights =[w for w in weights if w > 0]
    if pos_names:
        fig, ax = plt.subplots(figsize=(9, max(4, len(pos_names) * 0.6)))
        ax.barh(pos_names, pos_weights, color='#E74C3C', alpha=0.85)
        ax.set_title(f'LIME — Style B: Illicit-Pushing Features\n(Node {target_node_idx}, Example {example_id})',
                     fontsize=13, weight='bold')
        ax.set_xlabel('Feature Weight (positive = illicit)')
        plt.tight_layout()
        save_fig(f'lime_styleB_ex{example_id}.png', config)

    # Style C: Licit-pushing features
    neg_names   =[n for n, w in zip(names, weights) if w < 0]
    neg_weights =[abs(w) for w in weights if w < 0]
    if neg_names:
        fig, ax = plt.subplots(figsize=(9, max(4, len(neg_names) * 0.6)))
        ax.barh(neg_names, neg_weights, color='#2ECC71', alpha=0.85)
        ax.set_title(f'LIME — Style C: Licit-Pushing Features\n(Node {target_node_idx}, Example {example_id})',
                     fontsize=13, weight='bold')
        ax.set_xlabel('Feature Weight magnitude (negative = licit)')
        plt.tight_layout()
        save_fig(f'lime_styleC_ex{example_id}.png', config)


def explain_with_shap(wrapper, bg_data, target_feat, feature_names,
                      target_node_idx, example_id, config, nsamples=50):
    print(f"  Computing SHAP values (nsamples={nsamples})...")
    # Use kmeans background to speed up KernelExplainer
    background = shap.kmeans(bg_data, 20)
    explainer  = shap.KernelExplainer(wrapper, background)
    shap_vals  = explainer.shap_values(target_feat, nsamples=nsamples)

    # Robustly extract the shap vector for the target class/sample.
    # shap_vals may be:
    #  - a list of arrays (one per class), each shaped (n_samples, n_features)
    #  - a numpy array shaped (n_samples, n_features)
    #  - a numpy array shaped (n_classes, n_samples, n_features)
    # We'll prefer class index 1 (Illicit) if available, otherwise fallback to available data.
    try:
        if isinstance(shap_vals, list):
            # list of arrays per class
            if len(shap_vals) > 1:
                arr = np.asarray(shap_vals[1])   # class 1 (Illicit)
            else:
                arr = np.asarray(shap_vals[0])
            # arr expected shape: (n_samples, n_features)
            sv = arr[0]
        else:
            arr = np.asarray(shap_vals)
            if arr.ndim == 3:
                # (n_classes, n_samples, n_features)
                class_idx = 1 if arr.shape[0] > 1 else 0
                sv = arr[class_idx, 0, :]
            elif arr.ndim == 2:
                # could be (n_samples, n_features) or (n_classes, n_features)
                if arr.shape[0] == 1:
                    sv = arr[0]
                else:
                    # ambiguous: if number of rows equals number of classes? fallback to last row or class 1
                    class_idx = 1 if arr.shape[0] > 1 else 0
                    sv = arr[class_idx]
            else:
                # unexpected shape
                raise ValueError(f"Unexpected shap_values shape: {arr.shape}")
    except Exception as e:
        # As a last resort, try to flatten and take the first n features
        arr = np.asarray(shap_vals)
        arr_flat = arr.reshape(-1)
        n_feats = len(feature_names)
        if arr_flat.size >= n_feats:
            sv = arr_flat[:n_feats]
        else:
            raise RuntimeError(f"Unable to parse shap_values from explainer: {e}")

    # Ensure sv is 1D array of length n_features
    sv = np.asarray(sv).flatten()
    n_features = len(feature_names)
    if sv.size != n_features:
        # If lengths mismatch, try to align by taking first n_features or padding with zeros
        if sv.size > n_features:
            sv = sv[:n_features]
        else:
            sv = np.pad(sv, (0, n_features - sv.size), mode='constant', constant_values=0.0)

    # Get top 10 features by absolute SHAP value
    sorted_idx = np.argsort(np.abs(sv))[::-1][:10]
    top_names  =[feature_names[i] for i in sorted_idx]
    top_vals   = sv[sorted_idx]
    bar_colors =['#D62728' if v > 0 else '#1F77B4' for v in top_vals]

    # Style A: Bar chart
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top_names[::-1], top_vals[::-1], color=bar_colors[::-1], alpha=0.9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'SHAP — Style A: Bar Chart\n(Node {target_node_idx}, Example {example_id})',
                 fontsize=13, weight='bold')
    ax.set_xlabel('SHAP Value')
    legend =[mpatches.Patch(color='#D62728', label='→ Illicit (positive SHAP)'),
              mpatches.Patch(color='#1F77B4', label='→ Licit (negative SHAP)')]
    ax.legend(handles=legend)
    plt.tight_layout()
    save_fig(f'shap_styleA_ex{example_id}.png', config)

    # Style B: Waterfall chart
    cumulative = 0.0
    # For waterfall we want cumulative starting from baseline, so iterate in shown order
    cum_vals = []
    # We'll compute starts for each bar (left positions)
    for v in top_vals[::-1]:
        cum_vals.append(cumulative)
        cumulative += v
    # cum_vals currently corresponds to reversed top_vals; reverse to match top_names[::-1] usage
    cum_vals = list(reversed(cum_vals))

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, (name, val, start) in enumerate(zip(top_names[::-1], top_vals[::-1], cum_vals[::-1])):
        color = '#D62728' if val > 0 else '#1F77B4'
        ax.barh(i, val, left=start, color=color, alpha=0.85, height=0.6)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names[::-1], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'SHAP — Style B: Waterfall Chart\n(Node {target_node_idx}, Example {example_id})',
                 fontsize=13, weight='bold')
    ax.set_xlabel('Cumulative SHAP Value')
    plt.tight_layout()
    save_fig(f'shap_styleB_ex{example_id}.png', config)

    # Style C: Dot Lollipop
    dot_colors =['#D62728' if v > 0 else '#1F77B4' for v in top_vals[::-1]]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(top_vals[::-1], range(len(top_names)), c=dot_colors, s=120, zorder=3)
    for i, val in enumerate(top_vals[::-1]):
        ax.plot([0, val],[i, i], color='grey', linewidth=1, zorder=2)
    ax.set_yticks(range(len(top_names)))
    ax.set_yticklabels(top_names[::-1], fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'SHAP — Style C: Dot Lollipop\n(Node {target_node_idx}, Example {example_id})',
                 fontsize=13, weight='bold')
    ax.set_xlabel('SHAP Value')
    plt.tight_layout()
    save_fig(f'shap_styleC_ex{example_id}.png', config)


def explain_gnn_predictions(model, data, feature_names, config, num_examples=2):
    print("\nInitiating XAI Analysis...")
    model.eval()
    device = config['device']

    # Find true-positive illicit nodes from test set
    test_idx     = torch.where(data.test_mask)[0]
    test_sampler = SubgraphBatchSampler(
        test_idx, config['batch_size'] * 2, config['num_hops'],
        data.edge_index, data.num_nodes, shuffle=False)
    criterion = FocalLoss(config['focal_alpha'], config['focal_gamma'])
    _, _, y_true_arr, preds_arr, _ = run_epoch_eval(
        model, test_sampler, data, criterion, device)

    true_positives = [test_idx[i].item()
                      for i in range(min(len(test_idx), len(preds_arr)))
                      if y_true_arr[i] == 1 and preds_arr[i] == 1]

    if len(true_positives) == 0:
        print("No True Positives found — using random illicit nodes.")
        true_positives = [test_idx[i].item()
                          for i in range(len(test_idx))
                          if data.y[test_idx[i]].item() == 1]

    # Background: 100 licit training nodes
    train_idx = torch.where(data.train_mask)[0]
    bg_idx    = train_idx[data.y[train_idx] == 0][:100]
    bg_data   = data.x[bg_idx].cpu().numpy()

    for i in range(min(num_examples, len(true_positives))):
        target_node_idx = true_positives[i]
        print(f"\n--- Explaining Node {target_node_idx} (Example {i+1}) ---")
        target_feat = data.x[target_node_idx].cpu().numpy().reshape(1, -1)
        wrapper     = _build_xai_wrapper(model, data, target_node_idx, device, config['num_hops'])

        print("  Generating LIME explanations (3 styles)...")
        explain_with_lime(wrapper, bg_data, target_feat, feature_names,
                          target_node_idx, example_id=i + 1, config=config)

        print("  Generating SHAP explanations (3 styles)...")
        explain_with_shap(wrapper, bg_data, target_feat, feature_names,
                          target_node_idx, example_id=i + 1, config=config, nsamples=50)

        torch.cuda.empty_cache()
        gc.collect()

    # XAI: LIME (3 styles) + SHAP (3 styles) for 2 nodes
explain_gnn_predictions(model, data, feature_names, CONFIG, num_examples=5)


Initiating XAI Analysis...

--- Explaining Node 157127 (Example 1) ---
  Generating LIME explanations (3 styles)...
Saved: /kaggle/working/lime_styleA_ex1.png
Saved: /kaggle/working/lime_styleB_ex1.png
  Generating SHAP explanations (3 styles)...
  Computing SHAP values (nsamples=50)...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: /kaggle/working/shap_styleA_ex1.png
Saved: /kaggle/working/shap_styleB_ex1.png
Saved: /kaggle/working/shap_styleC_ex1.png

--- Explaining Node 157140 (Example 2) ---
  Generating LIME explanations (3 styles)...
Saved: /kaggle/working/lime_styleA_ex2.png
Saved: /kaggle/working/lime_styleB_ex2.png
  Generating SHAP explanations (3 styles)...
  Computing SHAP values (nsamples=50)...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: /kaggle/working/shap_styleA_ex2.png
Saved: /kaggle/working/shap_styleB_ex2.png
Saved: /kaggle/working/shap_styleC_ex2.png

--- Explaining Node 157158 (Example 3) ---
  Generating LIME explanations (3 styles)...
Saved: /kaggle/working/lime_styleA_ex3.png
Saved: /kaggle/working/lime_styleB_ex3.png
Saved: /kaggle/working/lime_styleC_ex3.png
  Generating SHAP explanations (3 styles)...
  Computing SHAP values (nsamples=50)...


...

Saved: /kaggle/working/shap_styleA_ex3.png
Saved: /kaggle/working/shap_styleB_ex3.png
Saved: /kaggle/working/shap_styleC_ex3.png

--- Explaining Node 157361 (Example 4) ---
  Generating LIME explanations (3 styles)...
Saved: /kaggle/working/lime_styleA_ex4.png
Saved: /kaggle/working/lime_styleB_ex4.png
  Generating SHAP explanations (3 styles)...
  Computing SHAP values (nsamples=50)...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: /kaggle/working/shap_styleA_ex4.png
Saved: /kaggle/working/shap_styleB_ex4.png
Saved: /kaggle/working/shap_styleC_ex4.png

--- Explaining Node 157454 (Example 5) ---
  Generating LIME explanations (3 styles)...
Saved: /kaggle/working/lime_styleA_ex5.png
Saved: /kaggle/working/lime_styleB_ex5.png
  Generating SHAP explanations (3 styles)...
  Computing SHAP values (nsamples=50)...


  0%|          | 0/1 [00:00<?, ?it/s]

Saved: /kaggle/working/shap_styleA_ex5.png
Saved: /kaggle/working/shap_styleB_ex5.png
Saved: /kaggle/working/shap_styleC_ex5.png
